In [23]:
%run Packages.ipynb


In [116]:
### Check broken triangles

def is_triangle(E,i,j,k):
    """ Check if a given 3 indecies make a triangle in E(G)"""
    return tuple(sorted([i,j])) in E and tuple(sorted([i,k])) in E and tuple(sorted([j,k])) in E


def metric_triangles_mtx(G):
    """Returns the metric testing matrix for broken triangles"""
    D = make_index_encoding(G)
    I = get_edge_inedcies(G,D)
    n = G.order()
    E = G.edges(sort=True,labels=False)
    e = G.size()
    M = np.zeros(e)
    changed = 0
    for T in combinations(range(e),3):
        if is_triangle(E,T[0],T[1],T[2]):
            changed = 1
            T_edges = list(combinations(T,2))
            for i in range(3):
                z = np.zeros(e)
                z[D[T_edges[(0+i)%3]]] = 1
                z[D[T_edges[(1+i)%3]]] = 1
                z[D[T_edges[(2+i)%3]]] = -1
                M = np.vstack((M,z))
    if not changed:
        return M
    M = np.delete(M,0,0)
    return M

### Generate the metric testing matrix for a graph G
def induced_cycle_matrix(G):
    m = G.num_edges()
    ind_enc = make_index_encoding(G)
    mtxs = list()
    count = 0
    for C in get_chordless_cycles(G):
        count +=1
        M = np.zeros(m)
        C_ind = [*map(ind_enc.get, C)] # All indiced of the relevant edges
        for ind in C_ind:
            newrow = np.zeros(m)
            newrow[C_ind] = 1
            newrow[ind] = -1
            M = np.vstack((M, newrow))
        mtxs.append(sparse.csc_matrix(M))
    if count == 0:
        return np.zeros(m),count
    Phi = mtxs[0]
    for i in range(1,len(mtxs)):
        Phi = sparse.vstack((Phi,mtxs[i]))
    return Phi, count
        
def get_list_of_edges(cyc_vtx):
    """ Given a list of vertices of a cycle in the graph, return all its edges"""
    k = len(cyc_vtx)
    return [tuple(sorted((cyc_vtx[i],cyc_vtx[(i+1) % k]))) for i in range(k)]

def get_chordless_cycles(G):
    g = G.copy()
    for C in nx.chordless_cycles(g.networkx_graph()):
        yield get_list_of_edges(C)



In [5]:
### Auxiliary stuff
def count_simple_cycles(G,kmin = 3, kmax = 0):
    if kmin < 3:
        kmin = 3
    
    if kmax > G.order() or kmax == 0:
        kmax = G.order()
    S = 0
    for k in range(kmin,kmax+1):
        H = graphs.CycleGraph(k)
        S += G.subgraph_search_count(H,induced = True)/H.automorphism_group(return_group=False,order=True)
    return S